# Assignment No. 4 — Diffusion Models for High-Resolution Image Generation & Reconstruction
**Course:** Generative AI (AI4009) | Spring 2026  
**Platform:** Kaggle | GPU: T4 x2

---
## Table of Contents
1. Environment Setup & Imports
2. Configuration
3. Data Preprocessing
4. Forward Diffusion Process
5. U-Net Architecture
6. Reverse Process & Training
7. Image Reconstruction
8. Image Generation
9. Visualization Module
10. Quantitative Evaluation (PSNR & SSIM)
11. Gradio App Deployment

## 1. Environment Setup & Imports

In [ ]:
# Install required libraries
!pip install -q gradio torchmetrics

In [ ]:
import os
import math
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import torchvision

# Metrics
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## 2. Configuration

In [ ]:
class Config:
    # Data
    DATA_PATH = '/kaggle/input/celebahq256-images-only/data256x256'  # CelebA-HQ path
    IMAGE_SIZE = 128          # Use 256 for higher quality (requires more VRAM)
    CHANNELS = 3
    
    # Diffusion
    TIMESTEPS = 300           # Number of diffusion steps (200-500 recommended)
    BETA_START = 1e-4
    BETA_END = 0.02
    SCHEDULE = 'cosine'       # 'linear' or 'cosine'
    
    # Training
    BATCH_SIZE = 16
    EPOCHS = 50
    LR = 2e-4
    NUM_WORKERS = 4
    GRAD_CLIP = 1.0
    MIXED_PRECISION = True
    
    # U-Net
    BASE_CHANNELS = 64        # Channel progression: 64 → 128 → 256
    CHANNEL_MULTS = (1, 2, 4)
    NUM_RES_BLOCKS = 2
    TIME_EMB_DIM = 256
    
    # Sampling
    SAMPLE_STEPS = 300        # Can reduce for faster sampling
    NUM_SAMPLES = 5
    
    # Paths
    SAVE_DIR = '/kaggle/working/outputs'
    CHECKPOINT_PATH = '/kaggle/working/ddpm_checkpoint.pth'

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
print('Configuration ready.')

## 3. Data Preprocessing

In [ ]:
class ImageDataset(Dataset):
    """Generic image dataset supporting .jpg, .jpeg, .png files."""
    
    EXTS = ['.jpg', '.jpeg', '.png', '.webp']
    
    def __init__(self, folder, image_size):
        super().__init__()
        self.folder = Path(folder)
        self.image_size = image_size
        self.paths = [
            p for p in self.folder.rglob('*')
            if p.suffix.lower() in self.EXTS
        ]
        print(f'Found {len(self.paths)} images in {folder}')
        
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Scale to [-1, 1]
        ])
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img)


def get_dataloader(cfg):
    dataset = ImageDataset(cfg.DATA_PATH, cfg.IMAGE_SIZE)
    loader = DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True
    )
    return loader, dataset


def unnormalize(x):
    """Convert [-1,1] tensors back to [0,1] for visualization."""
    return (x.clamp(-1, 1) + 1) / 2


dataloader, dataset = get_dataloader(cfg)

# Visualize some training samples
sample_batch = next(iter(dataloader))[:8]
grid = make_grid(unnormalize(sample_batch), nrow=4, padding=2)
plt.figure(figsize=(12, 6))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.title('Sample Training Images', fontsize=14)
plt.axis('off')
plt.savefig(f'{cfg.SAVE_DIR}/sample_training_images.png', bbox_inches='tight')
plt.show()

## 4. Forward Diffusion Process

In [ ]:
class DiffusionSchedule:
    """
    DDPM Noise Schedule.
    Supports linear and cosine schedules.
    Precomputes all diffusion constants for efficiency.
    """
    
    def __init__(self, cfg):
        self.T = cfg.TIMESTEPS
        self.device = device
        
        # --- Build beta schedule ---
        if cfg.SCHEDULE == 'linear':
            betas = self._linear_schedule(cfg.BETA_START, cfg.BETA_END, self.T)
        elif cfg.SCHEDULE == 'cosine':
            betas = self._cosine_schedule(self.T)
        else:
            raise ValueError(f'Unknown schedule: {cfg.SCHEDULE}')
        
        # --- Precompute diffusion constants ---
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)           # ᾱ_t
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)  # ᾱ_{t-1}
        
        self.register('betas', betas)
        self.register('alphas', alphas)
        self.register('alphas_cumprod', alphas_cumprod)
        self.register('alphas_cumprod_prev', alphas_cumprod_prev)
        
        # Forward process q(x_t | x_0)
        self.register('sqrt_alphas_cumprod', alphas_cumprod.sqrt())
        self.register('sqrt_one_minus_alphas_cumprod', (1 - alphas_cumprod).sqrt())
        self.register('log_one_minus_alphas_cumprod', (1 - alphas_cumprod).log())
        self.register('sqrt_recip_alphas_cumprod', (1 / alphas_cumprod).sqrt())
        self.register('sqrt_recipm1_alphas_cumprod', (1 / alphas_cumprod - 1).sqrt())
        
        # Posterior q(x_{t-1} | x_t, x_0)
        posterior_variance = betas * (1 - alphas_cumprod_prev) / (1 - alphas_cumprod)
        self.register('posterior_variance', posterior_variance)
        self.register('posterior_log_variance_clipped',
                      posterior_variance.clamp(min=1e-20).log())
        self.register('posterior_mean_coef1',
                      betas * alphas_cumprod_prev.sqrt() / (1 - alphas_cumprod))
        self.register('posterior_mean_coef2',
                      (1 - alphas_cumprod_prev) * alphas.sqrt() / (1 - alphas_cumprod))
    
    def register(self, name, tensor):
        setattr(self, name, tensor.float().to(self.device))
    
    @staticmethod
    def _linear_schedule(beta_start, beta_end, timesteps):
        return torch.linspace(beta_start, beta_end, timesteps)
    
    @staticmethod
    def _cosine_schedule(timesteps, s=0.008):
        """Cosine schedule as proposed in 'Improved DDPM'."""
        steps = timesteps + 1
        x = torch.linspace(0, timesteps, steps)
        alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return betas.clamp(0, 0.999)
    
    def _extract(self, arr, t, x_shape):
        """Extract values at timestep t and reshape for broadcasting."""
        batch_size = t.shape[0]
        out = arr.gather(-1, t)
        return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))
    
    def q_sample(self, x_start, t, noise=None):
        """Forward process: add noise to x_start at timestep t.
        q(x_t | x_0) = N(x_t; sqrt(ᾱ_t)*x_0, (1-ᾱ_t)*I)
        """
        if noise is None:
            noise = torch.randn_like(x_start)
        sqrt_alphas_cumprod_t = self._extract(self.sqrt_alphas_cumprod, t, x_start.shape)
        sqrt_one_minus_alphas_cumprod_t = self._extract(
            self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
        return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise
    
    def predict_start_from_noise(self, x_t, t, noise):
        """Reconstruct x_0 from x_t and predicted noise."""
        return (
            self._extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -
            self._extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise
        )
    
    def q_posterior(self, x_start, x_t, t):
        """Compute posterior mean and variance."""
        posterior_mean = (
            self._extract(self.posterior_mean_coef1, t, x_t.shape) * x_start +
            self._extract(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        posterior_variance = self._extract(self.posterior_variance, t, x_t.shape)
        posterior_log_variance = self._extract(
            self.posterior_log_variance_clipped, t, x_t.shape)
        return posterior_mean, posterior_variance, posterior_log_variance
    
    @torch.no_grad()
    def p_sample(self, model, x_t, t):
        """Single reverse step: sample x_{t-1} from p(x_{t-1} | x_t)."""
        model.eval()
        b = x_t.shape[0]
        t_tensor = torch.full((b,), t, device=self.device, dtype=torch.long)
        
        # Predict noise
        pred_noise = model(x_t, t_tensor)
        
        # Compute x_0 estimate
        x_start = self.predict_start_from_noise(x_t, t_tensor, pred_noise)
        x_start = x_start.clamp(-1, 1)
        
        # Get posterior parameters
        model_mean, _, model_log_variance = self.q_posterior(x_start, x_t, t_tensor)
        
        # Sample
        noise = torch.randn_like(x_t) if t > 0 else torch.zeros_like(x_t)
        return model_mean + (0.5 * model_log_variance).exp() * noise
    
    @torch.no_grad()
    def p_sample_loop(self, model, shape, return_intermediates=False):
        """Full reverse diffusion loop: pure noise → image."""
        model.eval()
        b = shape[0]
        img = torch.randn(shape, device=self.device)
        intermediates = [img.clone()]
        
        for t in tqdm(reversed(range(0, self.T)), desc='Sampling', total=self.T, leave=False):
            img = self.p_sample(model, img, t)
            if return_intermediates and t % (self.T // 5) == 0:
                intermediates.append(img.clone())
        
        intermediates.append(img.clone())
        return (img, intermediates) if return_intermediates else img


schedule = DiffusionSchedule(cfg)
print('Diffusion schedule ready.')
print(f'Schedule type: {cfg.SCHEDULE}')
print(f'Timesteps: {cfg.TIMESTEPS}')
print(f'Beta range: [{schedule.betas.min():.6f}, {schedule.betas.max():.6f}]')

In [ ]:
# Visualize Forward Diffusion — at least 5 noising steps
def visualize_forward_diffusion(dataset, schedule, cfg, num_steps=10):
    img = dataset[0].unsqueeze(0).to(device)  # (1, C, H, W)
    step_indices = torch.linspace(0, cfg.TIMESTEPS - 1, num_steps).long()
    
    fig, axes = plt.subplots(1, num_steps + 1, figsize=(2.5 * (num_steps + 1), 3))
    
    axes[0].imshow(unnormalize(img[0]).permute(1, 2, 0).cpu().numpy())
    axes[0].set_title('Original\nt=0', fontsize=9)
    axes[0].axis('off')
    
    for i, t in enumerate(step_indices):
        t_tensor = torch.tensor([t], device=device)
        noisy = schedule.q_sample(img, t_tensor)
        axes[i + 1].imshow(unnormalize(noisy[0]).permute(1, 2, 0).cpu().numpy())
        axes[i + 1].set_title(f't={t.item()}', fontsize=9)
        axes[i + 1].axis('off')
    
    plt.suptitle('Forward Diffusion Process (Progressive Noising)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{cfg.SAVE_DIR}/forward_diffusion.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('Forward diffusion visualization saved.')

visualize_forward_diffusion(dataset, schedule, cfg, num_steps=10)

## 5. U-Net Architecture

In [ ]:
# ─── Time Embedding ───────────────────────────────────────────────────────────

class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal timestep embeddings (from 'Attention Is All You Need')."""
    
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class TimeEmbedding(nn.Module):
    """Projects sinusoidal embedding through 2 linear layers with GELU."""
    
    def __init__(self, time_emb_dim):
        super().__init__()
        self.sinusoidal = SinusoidalPositionEmbeddings(time_emb_dim)
        self.proj = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 4),
            nn.GELU(),
            nn.Linear(time_emb_dim * 4, time_emb_dim)
        )
    
    def forward(self, t):
        return self.proj(self.sinusoidal(t))


# ─── Basic Building Blocks ────────────────────────────────────────────────────

class WeightStandardizedConv2d(nn.Conv2d):
    """Conv2d with weight standardization for improved training stability."""
    
    def forward(self, x):
        weight = self.weight
        mean = weight.mean(dim=[1, 2, 3], keepdim=True)
        var = weight.var(dim=[1, 2, 3], keepdim=True, unbiased=False)
        weight = (weight - mean) / (var + 1e-5).sqrt()
        return F.conv2d(x, weight, self.bias, self.stride, self.padding,
                        self.dilation, self.groups)


class ResidualBlock(nn.Module):
    """Residual block with GroupNorm and time-step conditioning."""
    
    def __init__(self, in_channels, out_channels, time_emb_dim, groups=8):
        super().__init__()
        
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels * 2)  # scale + shift
        )
        
        self.block1 = nn.Sequential(
            nn.GroupNorm(groups, in_channels),
            nn.SiLU(),
            WeightStandardizedConv2d(in_channels, out_channels, 3, padding=1)
        )
        
        self.block2 = nn.Sequential(
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(),
            nn.Dropout(0.1),
            WeightStandardizedConv2d(out_channels, out_channels, 3, padding=1)
        )
        
        self.residual_conv = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )
    
    def forward(self, x, time_emb):
        h = self.block1(x)
        
        # Time conditioning via adaptive group norm (scale & shift)
        time_cond = self.time_mlp(time_emb)  # (B, out_channels * 2)
        time_cond = time_cond[:, :, None, None]  # Broadcast over H, W
        scale, shift = time_cond.chunk(2, dim=1)
        h = h * (1 + scale) + shift
        
        h = self.block2(h)
        return h + self.residual_conv(x)


class AttentionBlock(nn.Module):
    """Self-attention block for capturing long-range dependencies."""
    
    def __init__(self, channels, num_heads=4, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, channels)
        self.attn = nn.MultiheadAttention(channels, num_heads, batch_first=True)
    
    def forward(self, x):
        b, c, h, w = x.shape
        x_norm = self.norm(x)
        # Reshape for attention: (B, C, H, W) → (B, H*W, C)
        x_flat = x_norm.view(b, c, -1).transpose(1, 2)
        attn_out, _ = self.attn(x_flat, x_flat, x_flat)
        attn_out = attn_out.transpose(1, 2).view(b, c, h, w)
        return x + attn_out  # Residual


class Downsample(nn.Module):
    """Learnable downsampling using strided convolution."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
    
    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    """Learnable upsampling using bilinear + conv."""
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)
    
    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.conv(x)


# ─── U-Net ────────────────────────────────────────────────────────────────────

class UNet(nn.Module):
    """
    Simplified U-Net for DDPM noise prediction.
    Architecture:
      - Encoder: Conv → [ResBlock × N + Downsample] × L
      - Bottleneck: ResBlock → Attention → ResBlock
      - Decoder: [Upsample + ResBlock × N] × L → Conv
    Channel progression: 64 → 128 → 256
    """
    
    def __init__(self, cfg):
        super().__init__()
        
        in_channels = cfg.CHANNELS
        base_c = cfg.BASE_CHANNELS
        mults = cfg.CHANNEL_MULTS
        num_res = cfg.NUM_RES_BLOCKS
        time_dim = cfg.TIME_EMB_DIM
        
        channel_dims = [base_c * m for m in mults]  # e.g. [64, 128, 256]
        
        # ── Time Embedding ──
        self.time_embed = TimeEmbedding(time_dim)
        
        # ── Initial Conv ──
        self.init_conv = nn.Conv2d(in_channels, base_c, 7, padding=3)
        
        # ── Encoder ──
        self.encoder_blocks = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        
        in_c = base_c
        skip_channels = [base_c]  # Track skip connection channel sizes
        
        for i, out_c in enumerate(channel_dims):
            level_blocks = nn.ModuleList()
            for _ in range(num_res):
                level_blocks.append(ResidualBlock(in_c, out_c, time_dim))
                skip_channels.append(out_c)
                in_c = out_c
            # Add attention at the deepest level
            if i == len(channel_dims) - 1:
                level_blocks.append(AttentionBlock(out_c))
            self.encoder_blocks.append(level_blocks)
            if i < len(channel_dims) - 1:
                self.downsamples.append(Downsample(out_c))
                skip_channels.append(out_c)
        
        # ── Bottleneck ──
        mid_c = channel_dims[-1]
        self.mid_block1 = ResidualBlock(mid_c, mid_c, time_dim)
        self.mid_attn = AttentionBlock(mid_c)
        self.mid_block2 = ResidualBlock(mid_c, mid_c, time_dim)
        
        # ── Decoder ──
        self.decoder_blocks = nn.ModuleList()
        self.upsamples = nn.ModuleList()
        
        for i, out_c in enumerate(reversed(channel_dims)):
            level_blocks = nn.ModuleList()
            for j in range(num_res + 1):
                skip_c = skip_channels.pop()
                level_blocks.append(ResidualBlock(in_c + skip_c, out_c, time_dim))
                in_c = out_c
            if i < len(channel_dims) - 1:
                level_blocks.append(AttentionBlock(out_c))
            self.decoder_blocks.append(level_blocks)
            if i < len(channel_dims) - 1:
                self.upsamples.append(Upsample(out_c))
        
        # ── Final Output ──
        self.final_norm = nn.GroupNorm(8, base_c)
        self.final_act = nn.SiLU()
        self.final_conv = nn.Conv2d(base_c, in_channels, 1)
    
    def forward(self, x, t):
        # Time embedding
        time_emb = self.time_embed(t)  # (B, time_dim)
        
        # Initial conv
        x = self.init_conv(x)
        
        # Encoder
        skips = [x]
        down_idx = 0
        
        for i, level_blocks in enumerate(self.encoder_blocks):
            for block in level_blocks:
                if isinstance(block, ResidualBlock):
                    x = block(x, time_emb)
                    skips.append(x)
                else:  # AttentionBlock
                    x = block(x)
            if i < len(self.encoder_blocks) - 1:
                x = self.downsamples[down_idx](x)
                skips.append(x)
                down_idx += 1
        
        # Bottleneck
        x = self.mid_block1(x, time_emb)
        x = self.mid_attn(x)
        x = self.mid_block2(x, time_emb)
        
        # Decoder
        up_idx = 0
        for i, level_blocks in enumerate(self.decoder_blocks):
            for block in level_blocks:
                if isinstance(block, ResidualBlock):
                    skip = skips.pop()
                    x = torch.cat([x, skip], dim=1)
                    x = block(x, time_emb)
                else:  # AttentionBlock
                    x = block(x)
            if i < len(self.decoder_blocks) - 1:
                x = self.upsamples[up_idx](x)
                up_idx += 1
        
        # Final output
        x = self.final_norm(x)
        x = self.final_act(x)
        return self.final_conv(x)


# Instantiate model
model = UNet(cfg).to(device)

# Use DataParallel if multiple GPUs
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs with DataParallel')
    model = nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Quick shape test
with torch.no_grad():
    dummy_x = torch.randn(2, cfg.CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE).to(device)
    dummy_t = torch.randint(0, cfg.TIMESTEPS, (2,)).to(device)
    dummy_out = model(dummy_x, dummy_t)
    print(f'Input shape: {dummy_x.shape} → Output shape: {dummy_out.shape}')

## 6. Reverse Process & Training

In [ ]:
def get_loss(model, schedule, x_start, t, noise=None):
    """DDPM training loss: MSE between predicted and actual noise."""
    if noise is None:
        noise = torch.randn_like(x_start)
    x_noisy = schedule.q_sample(x_start=x_start, t=t, noise=noise)
    predicted_noise = model(x_noisy, t)
    return F.mse_loss(predicted_noise, noise)


def train_one_epoch(model, dataloader, optimizer, schedule, scaler, cfg, epoch):
    model.train()
    total_loss = 0.0
    pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{cfg.EPOCHS}', leave=True)
    
    for step, batch in enumerate(pbar):
        x = batch.to(device)
        t = torch.randint(0, cfg.TIMESTEPS, (x.shape[0],), device=device).long()
        
        optimizer.zero_grad()
        
        if cfg.MIXED_PRECISION:
            with autocast():
                loss = get_loss(model, schedule, x, t)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = get_loss(model, schedule, x, t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(dataloader)


def save_checkpoint(model, optimizer, epoch, loss_history, cfg):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss_history': loss_history
    }, cfg.CHECKPOINT_PATH)


def load_checkpoint(model, optimizer, cfg):
    if os.path.exists(cfg.CHECKPOINT_PATH):
        ckpt = torch.load(cfg.CHECKPOINT_PATH, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        print(f"Resumed from epoch {ckpt['epoch']+1}")
        return ckpt['epoch'] + 1, ckpt['loss_history']
    return 0, []


# Setup optimizer and scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=1e-6)
scaler = GradScaler(enabled=cfg.MIXED_PRECISION)

# Load checkpoint if available
start_epoch, loss_history = load_checkpoint(model, optimizer, cfg)

print('Training setup complete.')
print(f'Optimizer: AdamW | LR: {cfg.LR} | Scheduler: CosineAnnealing')
print(f'Mixed Precision: {cfg.MIXED_PRECISION}')

In [ ]:
# ─── Main Training Loop ───────────────────────────────────────────────────────

print('=' * 60)
print('Starting Training')
print('=' * 60)

best_loss = float('inf')

for epoch in range(start_epoch, cfg.EPOCHS):
    avg_loss = train_one_epoch(model, dataloader, optimizer, schedule, scaler, cfg, epoch)
    scheduler.step()
    loss_history.append(avg_loss)
    
    print(f'Epoch [{epoch+1}/{cfg.EPOCHS}] | Avg Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}')
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        save_checkpoint(model, optimizer, epoch, loss_history, cfg)
        print(f'  → Checkpoint saved at epoch {epoch+1}')
    
    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), f'{cfg.SAVE_DIR}/best_model.pth')

print('\nTraining Complete!')
print(f'Best Loss: {best_loss:.4f}')

In [ ]:
# Plot Training Loss
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(loss_history) + 1), loss_history, 'b-o', markersize=3, linewidth=1.5, label='Train Loss')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Training Loss vs Epochs', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{cfg.SAVE_DIR}/training_loss.png', bbox_inches='tight', dpi=120)
plt.show()
print('Loss plot saved.')

## 7. Image Reconstruction (Core Task)

In [ ]:
# Load best model for inference
best_model_path = f'{cfg.SAVE_DIR}/best_model.pth'
if os.path.exists(best_model_path):
    state_dict = torch.load(best_model_path, map_location=device)
    model.load_state_dict(state_dict)
    print('Loaded best model weights.')
model.eval()


def reconstruct_image(target_img_tensor, model, schedule, cfg):
    """
    Reconstruction task:
    1. Noise the target image to step T/2 (partial noising)
    2. Denoise from that point back to step 0
    This is the 'img2img' / reconstruction approach.
    """
    model.eval()
    x = target_img_tensor.unsqueeze(0).to(device)  # (1, C, H, W)
    
    # Noise to intermediate timestep
    noise_level = cfg.TIMESTEPS // 2
    t = torch.tensor([noise_level - 1], device=device)
    noise = torch.randn_like(x)
    x_noised = schedule.q_sample(x, t, noise)
    
    intermediates = [x_noised.clone()]
    
    # Reverse from noise_level back to 0
    x_current = x_noised
    step_interval = max(1, noise_level // 5)
    
    for t_val in tqdm(reversed(range(0, noise_level)), desc='Reconstructing', total=noise_level):
        x_current = schedule.p_sample(model, x_current, t_val)
        if t_val % step_interval == 0:
            intermediates.append(x_current.clone())
    
    return x_current, intermediates


# Pick a target image
target_idx = 42  # Change index to try different targets
target_img = dataset[target_idx]  # (C, H, W) in [-1, 1]

with torch.no_grad():
    reconstructed, recon_intermediates = reconstruct_image(target_img, model, schedule, cfg)

# Display target vs reconstructed
fig, axes = plt.subplots(1, 2 + len(recon_intermediates), figsize=(3 * (2 + len(recon_intermediates)), 4))

axes[0].imshow(unnormalize(target_img).permute(1, 2, 0).cpu().numpy())
axes[0].set_title('Target Image', fontsize=10, fontweight='bold')
axes[0].axis('off')

for i, interm in enumerate(recon_intermediates):
    axes[i + 1].imshow(unnormalize(interm[0]).permute(1, 2, 0).cpu().numpy())
    axes[i + 1].set_title(f'Step {i}', fontsize=9)
    axes[i + 1].axis('off')

axes[-1].imshow(unnormalize(reconstructed[0]).permute(1, 2, 0).cpu().numpy())
axes[-1].set_title('Reconstructed', fontsize=10, fontweight='bold')
axes[-1].axis('off')

plt.suptitle('Image Reconstruction: Target → Noised → Reconstructed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{cfg.SAVE_DIR}/reconstruction.png', bbox_inches='tight', dpi=120)
plt.show()
print('Reconstruction saved.')

## 8. Image Generation from Pure Noise

In [ ]:
# Generate 5 new images from pure noise
print('Generating images from pure noise...')

with torch.no_grad():
    generated_imgs, gen_intermediates = schedule.p_sample_loop(
        model,
        shape=(cfg.NUM_SAMPLES, cfg.CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
        return_intermediates=True
    )

# Display generated images
fig, axes = plt.subplots(1, cfg.NUM_SAMPLES, figsize=(4 * cfg.NUM_SAMPLES, 4))
for i in range(cfg.NUM_SAMPLES):
    axes[i].imshow(unnormalize(generated_imgs[i]).permute(1, 2, 0).cpu().numpy())
    axes[i].set_title(f'Generated {i+1}', fontsize=10)
    axes[i].axis('off')

plt.suptitle('Generated Images from Pure Noise', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{cfg.SAVE_DIR}/generated_images.png', bbox_inches='tight', dpi=120)
plt.show()

# Save individual images
for i in range(cfg.NUM_SAMPLES):
    save_image(unnormalize(generated_imgs[i]), f'{cfg.SAVE_DIR}/gen_{i+1}.png')

print(f'{cfg.NUM_SAMPLES} generated images saved.')

## 9. Visualization Module

In [ ]:
def visualization_module(dataset, model, schedule, cfg, sample_idx=0):
    """
    Complete visualization showing:
    - Original image
    - 5 forward diffusion steps
    - 5 reverse diffusion steps
    - 5 generated images
    """
    model.eval()
    orig = dataset[sample_idx].unsqueeze(0).to(device)
    
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle('Complete Diffusion Model Visualization', fontsize=16, fontweight='bold', y=0.98)
    
    # ── Row 1: Forward Diffusion ──
    forward_steps = np.linspace(0, cfg.TIMESTEPS - 1, 6, dtype=int)
    for col, t_val in enumerate(forward_steps):
        ax = fig.add_subplot(3, 6, col + 1)
        t_tensor = torch.tensor([t_val], device=device)
        noisy = schedule.q_sample(orig, t_tensor)
        ax.imshow(unnormalize(noisy[0]).permute(1, 2, 0).cpu().numpy())
        ax.set_title(f'Forward\nt={t_val}', fontsize=8)
        ax.axis('off')
    
    # ── Row 2: Reverse Diffusion Steps ──
    with torch.no_grad():
        _, rev_intermediates = schedule.p_sample_loop(
            model,
            shape=(1, cfg.CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
            return_intermediates=True
        )
    
    # Pick 6 evenly spaced intermediates
    idxs = np.linspace(0, len(rev_intermediates) - 1, 6, dtype=int)
    for col, idx in enumerate(idxs):
        ax = fig.add_subplot(3, 6, 6 + col + 1)
        ax.imshow(unnormalize(rev_intermediates[idx][0]).permute(1, 2, 0).cpu().numpy())
        label = 'Noise' if idx == 0 else ('Final' if idx == idxs[-1] else f'Step {idx}')
        ax.set_title(f'Reverse\n{label}', fontsize=8)
        ax.axis('off')
    
    # ── Row 3: Generated Images ──
    with torch.no_grad():
        gen = schedule.p_sample_loop(
            model,
            shape=(6, cfg.CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)
        )
    
    for col in range(6):
        ax = fig.add_subplot(3, 6, 12 + col + 1)
        ax.imshow(unnormalize(gen[col]).permute(1, 2, 0).cpu().numpy())
        ax.set_title(f'Generated\n{col+1}', fontsize=8)
        ax.axis('off')
    
    # Row labels
    for row, label in enumerate(['Forward Diffusion (Noising)', 'Reverse Diffusion (Denoising)', 'Generated Images']):
        fig.text(0.01, 0.83 - row * 0.33, label, fontsize=11, fontweight='bold',
                 va='center', rotation='vertical')
    
    plt.tight_layout(rect=[0.03, 0, 1, 0.97])
    plt.savefig(f'{cfg.SAVE_DIR}/full_visualization.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('Full visualization saved.')


visualization_module(dataset, model, schedule, cfg)

## 10. Quantitative Evaluation (PSNR & SSIM)

In [ ]:
def compute_metrics(target_tensor, reconstructed_tensor):
    """
    Compute PSNR and SSIM between target and reconstructed images.
    Both tensors should be in [-1, 1] range, shape (1, C, H, W).
    """
    # Convert to [0, 1]
    target_01 = unnormalize(target_tensor).cpu()
    recon_01 = unnormalize(reconstructed_tensor).cpu()
    
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0)
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0)
    
    psnr_val = psnr_metric(recon_01, target_01).item()
    ssim_val = ssim_metric(recon_01, target_01).item()
    
    return psnr_val, ssim_val


# Evaluate over multiple test images
print('Computing PSNR & SSIM over test images...')
num_eval = min(20, len(dataset))
psnr_scores, ssim_scores = [], []

for idx in tqdm(range(num_eval), desc='Evaluating'):
    target = dataset[idx]
    with torch.no_grad():
        recon, _ = reconstruct_image(target, model, schedule, cfg)
    psnr_val, ssim_val = compute_metrics(
        target.unsqueeze(0).to(device), recon
    )
    psnr_scores.append(psnr_val)
    ssim_scores.append(ssim_val)

print('\n' + '='*50)
print('QUANTITATIVE EVALUATION RESULTS')
print('='*50)
print(f'PSNR — Mean: {np.mean(psnr_scores):.2f} dB | Std: {np.std(psnr_scores):.2f}')
print(f'SSIM — Mean: {np.mean(ssim_scores):.4f}    | Std: {np.std(ssim_scores):.4f}')
print('='*50)

# Bar chart of metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(num_eval), psnr_scores, color='steelblue', alpha=0.8)
ax1.axhline(np.mean(psnr_scores), color='red', linestyle='--', label=f'Mean={np.mean(psnr_scores):.2f}')
ax1.set_title('PSNR per Image', fontsize=12, fontweight='bold')
ax1.set_xlabel('Image Index')
ax1.set_ylabel('PSNR (dB)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(range(num_eval), ssim_scores, color='darkorange', alpha=0.8)
ax2.axhline(np.mean(ssim_scores), color='red', linestyle='--', label=f'Mean={np.mean(ssim_scores):.4f}')
ax2.set_title('SSIM per Image', fontsize=12, fontweight='bold')
ax2.set_xlabel('Image Index')
ax2.set_ylabel('SSIM')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Quantitative Evaluation: PSNR & SSIM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{cfg.SAVE_DIR}/metrics.png', bbox_inches='tight', dpi=120)
plt.show()

## 11. Gradio App Deployment

In [ ]:
import gradio as gr


def gradio_generate(num_images=1, show_steps=True):
    """
    Gradio callback: generate images from noise and return:
    - Final generated images
    - Grid showing intermediate denoising steps
    """
    model.eval()
    num_images = int(num_images)
    
    with torch.no_grad():
        generated, intermediates = schedule.p_sample_loop(
            model,
            shape=(num_images, cfg.CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
            return_intermediates=True
        )
    
    # ── Final generated images ──
    final_imgs = []
    for i in range(num_images):
        img_np = (unnormalize(generated[i]).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        final_imgs.append(Image.fromarray(img_np))
    
    # ── Intermediate steps grid ──
    steps_to_show = min(8, len(intermediates))
    idxs = np.linspace(0, len(intermediates) - 1, steps_to_show, dtype=int)
    
    fig, axes = plt.subplots(1, steps_to_show, figsize=(2.5 * steps_to_show, 3))
    if steps_to_show == 1:
        axes = [axes]
    
    for j, idx in enumerate(idxs):
        step_img = unnormalize(intermediates[idx][0]).permute(1, 2, 0).cpu().numpy()
        axes[j].imshow(step_img)
        axes[j].set_title(f'Step {idx}', fontsize=8)
        axes[j].axis('off')
    
    plt.suptitle('Denoising Steps', fontsize=11, fontweight='bold')
    plt.tight_layout()
    
    steps_path = f'{cfg.SAVE_DIR}/gradio_steps.png'
    plt.savefig(steps_path, bbox_inches='tight', dpi=100)
    plt.close()
    
    return final_imgs[0] if num_images == 1 else final_imgs, Image.open(steps_path)


def gradio_reconstruct(input_image):
    """Gradio callback: reconstruct an uploaded image."""
    if input_image is None:
        return None, None
    
    # Preprocess
    transform = transforms.Compose([
        transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    img_tensor = transform(Image.fromarray(input_image).convert('RGB'))
    
    with torch.no_grad():
        recon, recon_steps = reconstruct_image(img_tensor, model, schedule, cfg)
    
    recon_np = (unnormalize(recon[0]).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    
    # Side-by-side comparison
    orig_resized = np.array(Image.fromarray(input_image).resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)))
    comparison = np.concatenate([orig_resized, recon_np], axis=1)
    
    return Image.fromarray(recon_np), Image.fromarray(comparison)


# ── Build Gradio Interface ──
with gr.Blocks(title='DDPM Image Generator', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎨 DDPM Image Generator
    **Generative AI Assignment 4** — Diffusion Models for High-Resolution Image Generation
    """)
    
    with gr.Tabs():
        
        with gr.TabItem('🌟 Generate from Noise'):
            gr.Markdown('Start from pure random noise and generate a new image through the reverse diffusion process.')
            with gr.Row():
                gen_btn = gr.Button('Generate Image', variant='primary', scale=1)
            with gr.Row():
                gen_output = gr.Image(label='Generated Image', type='pil')
                steps_output = gr.Image(label='Denoising Steps (Intermediate States)', type='pil')
            gen_btn.click(fn=gradio_generate, inputs=[], outputs=[gen_output, steps_output])
        
        with gr.TabItem('🔄 Reconstruct Image'):
            gr.Markdown('Upload an image. The model will noise it and reconstruct it using the reverse diffusion process.')
            with gr.Row():
                upload_input = gr.Image(label='Upload Target Image', type='numpy')
            recon_btn = gr.Button('Reconstruct', variant='primary')
            with gr.Row():
                recon_output = gr.Image(label='Reconstructed Image', type='pil')
                compare_output = gr.Image(label='Side-by-Side Comparison (Original | Reconstructed)', type='pil')
            recon_btn.click(fn=gradio_reconstruct, inputs=[upload_input],
                            outputs=[recon_output, compare_output])
    
    gr.Markdown('> **Model:** DDPM with U-Net | **Dataset:** CelebA-HQ | **Framework:** PyTorch (base layers only)')

demo.launch(share=True, debug=False)
print('Gradio app launched!')

---
## Summary

| Component | Details |
|---|---|
| **Model** | DDPM with U-Net backbone |
| **Dataset** | CelebA-HQ (128×128 or 256×256) |
| **Timesteps** | 300 (cosine schedule) |
| **U-Net channels** | 64 → 128 → 256 |
| **Training** | AdamW + CosineAnnealing + Mixed Precision |
| **Loss** | MSE (predicted vs actual noise) |
| **Metrics** | PSNR & SSIM |
| **App** | Gradio (Generate + Reconstruct tabs) |